# 8장. ACP Server에 MCP 추가

## MCP 서버 출력

In [1]:
%%writefile acp_project/mcpserver.py

from colorama import Fore
from mcp.server.fastmcp import FastMCP
import json 
import requests

mcp = FastMCP("hospitalserver")
    
# 서버 함수 빌드
@mcp.tool()
def list_hospital(town:str) -> str:
    """이 도구는 특정 동(town)에 있는 병원 목록을 반환합니다.
    Args:
        town: 검색할 동 이름 (예: "역삼동").

    Returns:
        str: 해당 동에 위치한 병원 목록
        """
    
    url = 'https://raw.githubusercontent.com/no-wave/llm-master-acp-cookbook/main/hospital_info.json'

    resp = requests.get(url)
    hospitals = json.loads(resp.text)

    # BUG FIX: docstring의 매개변수명(state->town)을 실제 함수와 일치시켰습니다.
    matches = [hosp for hosp in hospitals.values() if hosp['address']['town'] == town]    
    return str(matches) 

# 파일이 실행될 경우 서버 시작
if __name__ == "__main__":
    mcp.run(transport="stdio")

Overwriting acp_project/mcpserver.py


## MCP를 사용하도록 ACP 서버 업데이트

In [2]:
%%writefile acp_project/guideagent_server.py

from collections.abc import AsyncGenerator
from acp_sdk.models import Message, MessagePart
from acp_sdk.server import RunYield, RunYieldResume, Server
from smolagents import CodeAgent, DuckDuckGoSearchTool, LiteLLMModel, VisitWebpageTool, ToolCallingAgent, ToolCollection
from mcp import StdioServerParameters

server = Server()

model = LiteLLMModel(
    model_id="openai/gpt-4o-mini",  
    max_tokens=2048
)

server_parameters = StdioServerParameters(
    command="uv",
    args=["run", "mcpserver.py"],
    env=None,
)

@server.agent()
async def guide_search_agent(input: list[Message]) -> AsyncGenerator[RunYield, RunYieldResume]:
    """응급환자 발생시 관련 질문을 검색하여 지원하는 Agent다. 교내의 응급 발생 시 교직원과 학생이 응급 처치 방법을 검색하여 찾는 데 사용할 수 있다."""
    agent = CodeAgent(tools=[DuckDuckGoSearchTool(), VisitWebpageTool()], model=model)

    prompt = input[0].parts[0].content
    response = agent.run(prompt)

    yield Message(parts=[MessagePart(content=str(response))])

@server.agent()
async def hospital_agent(input: list[Message]) -> AsyncGenerator[RunYield, RunYieldResume]:
    """이것은 사용자가 응급환자 발생 시 근처 병원을 찾는 데 도움을 주는 에이전트다."""
    with ToolCollection.from_mcp(server_parameters, trust_remote_code=True) as tool_collection:
        agent = ToolCallingAgent(tools=[*tool_collection.tools], model=model)
        prompt = input[0].parts[0].content
        response = agent.run(prompt)
        
        yield Message(parts=[MessagePart(content=str(response))])

if __name__ == "__main__":
    server.run(port=8001)

Overwriting acp_project/guideagent_server.py


## 터미널에서 ACP x MCP 서버 실행

```
uv run guideagent_server.py
```

## ACPxMCP 서버 호출

In [3]:
import asyncio
import nest_asyncio
from acp_sdk.client import Client
from colorama import Fore

nest_asyncio.apply()

In [4]:
async with Client(base_url="http://localhost:8001") as hospital:
    run1 = await hospital.run_sync(
        agent="hospital_agent", input="저는 역삼동에 거주합니다. 근처에 병원이 있나요?"
    )

    if run1.output and run1.output[0].parts:
        content = run1.output[0].parts[0].content
        print(Fore.LIGHTMAGENTA_EX + content + Fore.RESET)
    else:
        print("출력 데이터가 비어있습니다. Agent 실행 결과를 확인하세요.")

역삼동에 위치한 병원 목록은 다음과 같습니다:

1. 247외과의원 - 서울특별시 강남구 봉은사로 108, 2층 (역삼동) - 전화: 02-562-0247
2. 365삼성의원 - 서울특별시 강남구 도곡로 331, 7층 (역삼동) - 전화: 02-555-0365
3. 47이끌의원 - 서울특별시 강남구 강남대로84길 6, 5층 (역삼동) - 전화: 02-2088-4711
4. SKY(스카이)치과의원 - 서울특별시 강남구 테헤란로26길 10, 성보빌딩 3층 (역삼동) - 전화: 02-3452-2828
...
(총 120개 이상의 병원 목록이 있습니다.)


In [6]:
async def run_workflow() -> None:
    async with Client(base_url="http://localhost:8001") as hospital:
        run1 = await hospital.run_sync(
            agent="hospital_agent", input="나는 역삼동에 살아. 근처에 병원이 있으면 찾아줄래?"
        )
        content = run1.output[0].parts[0].content
        print(Fore.LIGHTMAGENTA_EX+ content + Fore.RESET)

asyncio.run(run_workflow())

역삼동 근처에 있는 병원의 목록입니다:

1. **247외과의원**  
   - 주소: 서울특별시 강남구 봉은사로 108, 2층 (역삼동)  
   - 전화: 02-562-0247

2. **365삼성의원**  
   - 주소: 서울특별시 강남구 도곡로 331, 7층 (역삼동)  
   - 전화: 02-555-0365

3. **47이끌의원**  
   - 주소: 서울특별시 강남구 강남대로84길 6, 5층 (역삼동)  
   - 전화: 02-2088-4711

4. **SKY(스카이)치과의원**  
   - 주소: 서울특별시 강남구 테헤란로26길 10, 성보빌딩 3층 (역삼동)  
   - 전화: 02-3452-2828

5. **가뿐한의원**  
   - 주소: 서울특별시 강남구 테헤란로 322, 한신인터밸리24 지하2층 B102, B103호 (역삼동)  
   - 전화: 02-2183-1755

6. **가은치유한의원**  
   - 주소: 서울특별시 강남구 선릉로 427, 7층 (역삼동)  
   - 전화: 02-557-8845

7. **강남12의원**  
   - 주소: 서울특별시 강남구 테헤란로 111, 4층 (역삼동, 대건빌딩)  
   - 전화: 02-567-3000

8. **강남고운세상피부과의원**  
   - 주소: 서울특별시 강남구 테헤란로2길 8, 3층 (역삼동)  
   - 전화: 02-3477-2020

9. **강남닥터에버스의원**  
   - 주소: 서울특별시 강남구 강남대로 458, 남영빌딩 8~9층 (역삼동)  
   - 전화: 02-2138-0777

10. **강남하나로치과의원**  
   - 주소: 서울특별시 강남구 테헤란로 326, 역삼 아이타워 8층일부 (역삼동)  
   - 전화: 02-590-1313

이외에도 많은 병원이 가까이에 있습니다. 필요에 따라 추가 정보를 요청하실 수 있습니다.
